**문제6) 이미지 분류**

CIFAR-100 dataset 사용    ( 얘 말고 다른 데이터를 이용해도 됨. 음식, 사무용 집기, 라면(우동) ... )
특징
 - 클래스 수: 100개 (예: 사과, 버스, 산, 고래, 시계 등)
 - 샘플 수: 60,000장, 학습용(train): 50,000장, 테스트용(test): 10,000장
 - 이미지 크기: 32×32 RGB (작은 해상도)
 - 레이블 구조: 100개 fine labels (세부 클래스), 20개 coarse labels (상위 클래스 그룹)

 기본 CNN으로도 학습 가능하지만, 성능을 높이려면
 - 데이터 증강(ImageDataGenerator / tf.image)
 - 전이학습(사전학습 모델)
 - 정규화/드롭아웃/배치정규화 등을 함께 쓰는 게 효과적

-- 전체 흐름 요약 --

  작업1 :  CIFAR-100 dataset  분류 모델 작성 (MovileNetV2 모델로 전이학습, 파인튜닝)
  작업2 : 작성한  분류 모델 사용

              웹 브라우저에서 이미지 선택
                   → 장고 웹서버에 저장 → 서버 내부에서 시각화로 확인(matplotlib) + 딥러닝 분류  
                  → 클라이언트에 분류 결과만 반환하기

작업2를 좀더 구체적으로 보면
 1) 클라이언트
    : index.html에서 파일선택 버튼을 눌러 로컬 컴퓨터의 이미지 파일을 선택하고 화면에 선택된 이미지 출력
    : 분류결과요청 버튼 클릭 → AJAX 전송 (axios 모듈 사용)
 2) 서버(Django)
    : 수신된 이미지 파일 저장 → PIL + Matplotlib(imshow)으로 확인 → 딥러닝 분류 모델로 추론
    : 응답(JSON): 분류 결과만 반환(예 : bus)
 3) 클라이언트
    : 기존 이미지 아래에 이미지 분류 결과 문자열을 화면에 출력

In [1]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# CIFAR-100 dataset 읽기

(train_ds, val_ds, test_ds), metadata = tfds.load(
    'cifar100',
    split=['train[:80%]', 'train[80%:]', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True
)

ds_info = metadata

for image, label in train_ds.take(1):
    print(type(image), type(label))
    print(image.shape)
    print(label)

print(ds_info.features['label'].names)
print('class 수 : ', ds_info.features['label'].num_classes)

class_names = ds_info.features['label'].names
NUM_CLASSES = ds_info.features['label'].num_classes

2026-05-15 17:20:58.845765: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-15 17:20:58.942039: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-15 17:20:58.942569: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-15 17:20:59.062473: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-15 17:21:00.600586: W tensorflow/compiler/tf

Dl Completed...: 0 url [00:00, ? url/s]
Dl Completed...: 100%|██████████| 1/1 [01:13<00:00, 71.89s/ url]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]2026-05-15 17:22:17.562124: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-15 17:22:17.663736: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-05-15 17:22:17.664200: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), bu

Dataset cifar100 downloaded and prepared to /home/pnuav/tensorflow_datasets/cifar100/3.0.2. Subsequent calls will reuse this data.
<class 'tensorflow.python.framework.ops.EagerTensor'> <class 'tensorflow.python.framework.ops.EagerTensor'>
(32, 32, 3)
tf.Tensor(66, shape=(), dtype=int64)
['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel', 'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock', 'cloud', 'cockroach', 'couch', 'crab', 'crocodile', 'cup', 'dinosaur', 'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster', 'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion', 'lizard', 'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain', 'mouse', 'mushroom', 'oak_tree', 'orange', 'orchid', 'otter', 'palm_tree', 'pear', 'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine', 'possum', 'rabbit', 'raccoon', 'ray', 'road', 'roc

2026-05-15 17:23:00.550972: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2026-05-15 17:23:00.551365: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [2]:
# 전처리
IMG_SIZE = (256, 256)
BATCH_SIZE = 32

def preprocessFunc(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_ds = train_ds.map(preprocessFunc).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocessFunc).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(preprocessFunc).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

for image, label in train_ds.take(1):
    print(image.shape)
    print(label.shape)

(32, 256, 256, 3)
(32,)


2026-05-15 17:23:00.760088: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.
2026-05-15 17:23:00.760605: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [3]:
# 데이터 증강
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.05),
    layers.RandomContrast(0.05)
])

# 백본(base model) 불러오기
base_model = MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

/tmp/ipykernel_5406/633403514.py:10: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


In [4]:
# 모델 구성
model = models.Sequential([
    layers.Input(shape=IMG_SIZE + (3,)),
    data_augmentation,
    layers.Rescaling(2.0, offset=-1.0),
    base_model,
    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(NUM_CLASSES, activation='softmax')
])

print(model.summary())

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, verbose=1)
]

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 8, 8, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 100)            │        12,900 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,633,252 (10.05 MB)

 Trainable params: 374,500 (1.43 MB)

 Non-trainable params: 2,258,752 (8.62 MB)

None


In [5]:
# 전이학습 학습
history_baseline = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks,
    verbose=1
)

baseline_loss, baseline_acc = model.evaluate(test_ds)
print(f"transfer learning : loss:{baseline_loss: .4f}, acc:{baseline_acc: .4f}")

Epoch 1/50


2026-05-15 17:23:10.045110: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907


1250/1250 ━━━━━━━━━━━━━━━━━━━━ 550s 431ms/step - accuracy: 0.3679 - loss: 2.4755 - val_accuracy: 0.5127 - val_loss: 1.7559 - learning_rate: 0.0010
Epoch 2/50
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 535s 428ms/step - accuracy: 0.4928 - loss: 1.8618 - val_accuracy: 0.5554 - val_loss: 1.5875 - learning_rate: 0.0010
Epoch 3/50
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 538s 430ms/step - accuracy: 0.5218 - loss: 1.7391 - val_accuracy: 0.5736 - val_loss: 1.5137 - learning_rate: 0.0010
Epoch 4/50
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 536s 429ms/step - accuracy: 0.5396 - loss: 1.6605 - val_accuracy: 0.5738 - val_loss: 1.5088 - learning_rate: 0.0010
Epoch 5/50
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 0s 349ms/step - accuracy: 0.5476 - loss: 1.6150

KeyboardInterrupt: 

In [ ]:
# Fine Tuning
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print('미세 조정 시작 .....')

# Fine Tuning 학습
history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks,
    verbose=1
)

finetune_loss, finetune_acc = model.evaluate(test_ds)

print(f"fine tuning : loss:{finetune_loss: .4f}, acc:{finetune_acc: .4f}")

In [ ]:
# 전이학습 + 미세조정 결과 시각화
acc = history_baseline.history['accuracy'] + history_finetune.history['accuracy']
val_acc = history_baseline.history['val_accuracy'] + history_finetune.history['val_accuracy']
loss = history_baseline.history['loss'] + history_finetune.history['loss']
val_loss = history_baseline.history['val_loss'] + history_finetune.history['val_loss']
epoch_range = range(len(acc))


plt.figure(figsize=(14, 5))
# accuracy
plt.subplot(1, 2, 1)
plt.plot(epoch_range, acc, label='Train Acc')
plt.plot(epoch_range, val_acc, label='Val Acc')
plt.legend(loc='lower right')
plt.title('Train and Validation Accuracy')

# loss
plt.subplot(1, 2, 2)
plt.plot(epoch_range, loss, label='Train Loss')
plt.plot(epoch_range, val_loss, label='Val Loss')
plt.legend(loc='upper right')
plt.title('Train and Validation Loss')

plt.show()

In [ ]:
model.save('cifar100_mobilenetv2.keras')
print('keras 모델 저장 완료')